# Linear stability phase diagram

This notebook plots the linear-stability boundary in the parameter plane of inhibitory coupling strength $g$ and normalized external input $\nu_{\mathrm{ext}}/\nu_\theta$.

It loads previously computed numerical solutions of the characteristic equation from MATLAB `.mat` files. At each parameter combination, it takes the largest real part among the roots found numerically. The zero contour of this quantity is used to estimate the boundary between linearly stable and unstable regimes.

**Inputs.** The directory `DATA_DIR` must contain files named `Theoretical_prediction_<g>.mat`, with the value of $g$ encoded in each filename. Every file must contain:

- `list_varying_param`: the values of $\nu_{\mathrm{ext}}/\nu_\theta$;
- `local_minima_all`: a MATLAB cell array of numerical roots, with one entry per input value. Each nonempty entry is an array with three columns; the first column contains the real part of the root.

All files must share the same input grid. Empty or nonfinite root entries are ignored, leaving missing parameter combinations as `NaN`.

**Output.** The zero contour is plotted and saved to `stability_boundary.svg`. This notebook does **not** solve the characteristic equation or check whether all potentially dominant roots were found. It requires NumPy, SciPy, and Matplotlib.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.io import loadmat

# Directory containing the precomputed stability results
DATA_DIR = Path("stability_results")
FILE_PREFIX = "Theoretical_prediction_"
FILE_PATTERN = f"{FILE_PREFIX}*.mat"

plt.rcParams.update({"font.size": 12, "svg.fonttype": "none"})


In [ ]:
# Read input files and sort them by the numeric value of g.
file_list = list(DATA_DIR.glob(FILE_PATTERN))
if not file_list:
    raise FileNotFoundError(
        f"No files matching {FILE_PATTERN!r} in {DATA_DIR.resolve()}"
    )

files_with_g = sorted(
    ((float(path.stem[len(FILE_PREFIX):]), path) for path in file_list),
    key=lambda item: item[0],
)

g_values = []
input_reference = None
heatmap_rows = []

for g, path in files_with_g:
    data = loadmat(path, squeeze_me=False)
    minima_cells = data["local_minima_all"].ravel(order="F")
    varying_param = np.asarray(
        data["list_varying_param"], dtype=float
    ).ravel(order="F")

    if len(minima_cells) != len(varying_param):
        raise ValueError(
            f"{path.name}: number of cells and parameters differ."
        )

    # Maximum real part among roots found for each external-input value.
    leading_real = np.full(len(varying_param), np.nan)
    for j, minima in enumerate(minima_cells):
        minima = np.asarray(minima, dtype=float)
        if minima.size == 0:
            continue
        if minima.ndim != 2 or minima.shape[1] != 3:
            raise ValueError(
                f"{path.name}: expected minima with shape (n_roots, 3)."
            )

        valid = np.all(np.isfinite(minima), axis=1)
        if np.any(valid):
            leading_real[j] = np.max(minima[valid, 0])

    # Sort by external input and ensure that all files share its grid.
    order = np.argsort(varying_param)
    inputs = varying_param[order]
    leading_real = leading_real[order]

    if input_reference is None:
        input_reference = inputs
    elif (
        len(inputs) != len(input_reference)
        or not np.allclose(inputs, input_reference)
    ):
        raise ValueError(
            f"{path.name}: nu_ext/nu_theta grid differs from the other files."
        )

    g_values.append(g)
    heatmap_rows.append(leading_real)

g_values = np.asarray(g_values)
input_values = input_reference

# Shape: (n_input, n_g); rows correspond to input, columns to g.
max_real_map = np.asarray(heatmap_rows).T


In [ ]:
# Zero contour of the largest numerically found Re(lambda).
fig, ax = plt.subplots(figsize=(6, 4.5))

ax.contour(
    g_values,
    input_values,
    max_real_map,
    levels=[0],
    colors="black",
    linewidths=1.5,
)

ax.set_xlabel(r"$g$")
ax.set_ylabel(r"$\nu_{\mathrm{ext}}/\nu_{\theta}$")
ax.set_xlim(2.6, g_values.max())
ax.set_ylim(0.65, 1.6)
ax.set_box_aspect(1)

fig.tight_layout()
fig.savefig("stability_boundary.svg", format="svg", transparent=True)
plt.show()
